# Model Building

## Context
This is a **multivariate dataset** containing multiple clinical variables used for analyzing and predicting heart disease.

- Total attributes available: **76**
- Commonly used subset: **14 key features**
- Most widely used version: **Cleveland dataset**
- Primary task:  
  → Predict whether a patient has **heart disease** based on clinical measurements  
- Secondary task:  
  → Perform **exploratory analysis** to uncover patterns and insights

---

## Dataset Structure

### Key Features (14 Variables)

| Feature | Description |
|--------|------------|
| `age` | Age of the patient (years) |
| `sex` | Biological sex (Male/Female) |
| `cp` | Chest pain type *(typical angina, atypical angina, non-anginal, asymptomatic)* |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl *(True/False)* |
| `restecg` | Resting ECG results *(normal, ST-T abnormality, LV hypertrophy)* |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina *(True/False)* |
| `oldpeak` | ST depression (exercise vs rest) |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels (0–3) |
| `thal` | Thalassemia *(normal, fixed defect, reversible defect)* |
| `num` | Target variable (heart disease presence) |

---

### Additional Columns

| Feature | Description |
|--------|------------|
| `id` | Unique patient identifier |
| `origin` | Location of study |

---

## Target Variable

- `num` → Indicates **presence of heart disease**
  - `0` = No disease  
  - `1+` = Presence of disease  

---

## Use Cases

- Classification modeling (e.g., Logistic Regression, XGBoost)
- Risk prediction tools
- Clinical feature importance analysis
- Bias and fairness analysis across patient groups

---

## Data Sources

- Hungarian Institute of Cardiology (Budapest)
- University Hospital Zurich
- University Hospital Basel
- Cleveland Clinic Foundation

---

## Key References

- Detrano et al. (1989) – *American Journal of Cardiology*
- Aha & Kibler – Instance-based prediction methods
- Gennari et al. (1989) – Concept formation models

---

In [79]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix

In [60]:
df = pd.read_csv('../data/raw/heart_disease_uci.csv')
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [61]:
df.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


Check for missing values

In [62]:
df.isna().sum().sort_values(ascending=False)

ca          611
thal        486
slope       309
fbs          90
oldpeak      62
trestbps     59
exang        55
thalch       55
chol         30
restecg       2
cp            0
dataset       0
id            0
age           0
sex           0
num           0
dtype: int64

Drop ca, thal, slope, as they have large numbers of missing values. Imputating here would be futile.

In [63]:
df = df.drop(columns=["ca", "thal", "slope"])

Drop remaining missing values

In [64]:
df = df.dropna()

In [65]:
df.isna().sum().sort_values(ascending=False)

id          0
age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
num         0
dtype: int64

Create a binary healthy vs diseased variable. Check distribution

In [66]:
df["target"] = (df["num"] > 0).astype(int)
df["target"].value_counts()

target
1    383
0    357
Name: count, dtype: int64

Encode our binary variables so our models can handle them

In [67]:
df["sex"] = df["sex"].map({"Male": 1, "Female": 0})
df["fbs"] = df["fbs"].map({True: 1, False: 0})
df["exang"] = df["exang"].map({True: 1, False: 0})

For our multi category variables, we'll use one hot encoding

In [69]:
df = pd.get_dummies(df, columns=["cp", "restecg"], drop_first=True)

In [70]:
for col in df.columns:
    if df[col].dtype == "bool":
        df[col] = df[col].astype(int)

In [71]:
df = df.drop(columns=["dataset", "id"])

In [72]:
df.dtypes

age                           int64
sex                           int64
trestbps                    float64
chol                        float64
fbs                           int64
thalch                      float64
exang                         int64
oldpeak                     float64
num                           int64
target                        int64
cp_atypical angina            int64
cp_non-anginal                int64
cp_typical angina             int64
restecg_normal                int64
restecg_st-t abnormality      int64
dtype: object

Drop our target column (and num) as to setup prediction and eliminate leakage

In [73]:
X = df.drop(columns=['num', 'target'])
y = df['target']

Create our train test split, using 80/20

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [75]:
print(X)

     age  sex  trestbps   chol  fbs  thalch  exang  oldpeak  \
0     63    1     145.0  233.0    1   150.0      0      2.3   
1     67    1     160.0  286.0    0   108.0      1      1.5   
2     67    1     120.0  229.0    0   129.0      1      2.6   
3     37    1     130.0  250.0    0   187.0      0      3.5   
4     41    0     130.0  204.0    0   172.0      0      1.4   
..   ...  ...       ...    ...  ...     ...    ...      ...   
913   62    1     158.0  170.0    0   138.0      1      0.0   
914   46    1     134.0  310.0    0   126.0      0      0.0   
915   54    0     127.0  333.0    1   154.0      0      0.0   
917   55    1     122.0  223.0    1   100.0      0      0.0   
919   62    1     120.0  254.0    0    93.0      1      0.0   

     cp_atypical angina  cp_non-anginal  cp_typical angina  restecg_normal  \
0                     0               0                  1               0   
1                     0               0                  0               0   
2        

Fit a logistic regression model and find predicitons

In [76]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

Generate ROC AUC scores

In [78]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

Accuracy: 0.8445945945945946
ROC AUC: 0.91324200913242


An accuracy score of 84.45%, meaning 84.45% of patients are classified correctly. As well as an excellent AUC score of 91.32%, which means given a healthy and diseased patient, the model will rank them correctly 91.32% of the time. This is good! But we can dig deeper. How is the model making decisions?

In [80]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[59, 16],
       [ 7, 66]])

Sensitivity: TP / (TP + FN) = 66 / (66 + 7) ≈ 0.90 (90% of diseased patients detected)

Specificity: TN / (TN + FP) = 59 / (59 + 16) ≈ 0.79 (79% of healthy patients correctly identified)

A look into which coeffecients the model is using:

In [81]:
coeffs = pd.Series(model.coef_[0], index=X.columns)
coeffs.sort_values()

cp_atypical angina         -1.577074
cp_typical angina          -1.189912
cp_non-anginal             -1.106060
restecg_st-t abnormality   -0.210606
restecg_normal             -0.165548
thalch                     -0.013612
chol                       -0.002347
trestbps                    0.004785
age                         0.018076
fbs                         0.532542
oldpeak                     0.537969
exang                       1.028577
sex                         1.310187
dtype: float64